## Example 1: Basic DNA sequence analysis

In [ ]:
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction, molecular_weight
import Bio

# Example 1: Basic DNA sequence analysis with Biopython
print("Example 1: Basic DNA sequence analysis with Biopython")

try:
    # Create DNA sequence
    dna_seq = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
    print(f"✓ DNA sequence created: {dna_seq}")
    print(f"Length: {len(dna_seq)} nucleotides")
    
    # Basic sequence operations
    complement = dna_seq.complement()
    reverse_complement = dna_seq.reverse_complement()
    
    print(f"Complement: {complement}")
    print(f"Reverse complement: {reverse_complement}")
    
    # GC content analysis (using gc_fraction and converting to percentage)
    gc_content = gc_fraction(dna_seq) * 100
    print(f"GC content: {gc_content:.2f}%")
    
    # Transcription (DNA to RNA)
    rna_seq = dna_seq.transcribe()
    print(f"RNA sequence: {rna_seq}")
    
    # Translation (RNA to protein)
    protein_seq = rna_seq.translate()
    print(f"Protein sequence: {protein_seq}")
    
    # Molecular weight calculation
    dna_mw = molecular_weight(dna_seq, seq_type="DNA")
    print(f"DNA molecular weight: {dna_mw:.1f} Da")
    
    # Handle protein molecular weight (remove stop codons)
    protein_clean = str(protein_seq).replace('*', '')
    if protein_clean:
        protein_mw = molecular_weight(Seq(protein_clean), seq_type="protein")
        print(f"Protein molecular weight: {protein_mw:.1f} Da")
    else:
        print("Protein sequence contains only stop codons")
    
    print("Biopython DNA sequence analysis completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"biopython example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Protein analysis and properties

In [ ]:
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Seq import Seq

# Example 2: Protein analysis and properties
print("Example 2: Protein analysis and properties")

try:
    # Create protein sequence (without stop codons for analysis)
    protein_seq = Seq("MAEGEITTFTALTEKFNLPPGNYKKPKLLYCSNGGHFLRILPDGTVDGTRDRSDQHIQLQLSAESVGEVYIKSTETGQYLAMDTSGLLYGSQTPSEECLFLERLEENHYNTYTSKKHAEKNWFVGLKKNGSCKRGPRTHYGQKAILFLPLPV")
    
    # Create SeqRecord for better annotation
    protein_record = SeqRecord(protein_seq, id="sample_protein", description="Sample protein for analysis")
    
    print(f"✓ Protein sequence: {len(protein_seq)} amino acids")
    print(f"Description: {protein_record.description}")
    
    # Detailed protein analysis
    analyzer = ProteinAnalysis(str(protein_seq))
    
    # Basic properties
    molecular_weight = analyzer.molecular_weight()
    print(f"Molecular weight: {molecular_weight:.2f} Da")
    
    # Amino acid composition
    # amino_acids_percent maps each residue letter to its share of the sequence, in percent
    aa_composition = analyzer.amino_acids_percent
    print("Amino acid composition (top 5):")
    sorted_aa = sorted(aa_composition.items(), key=lambda x: x[1], reverse=True)[:5]
    for aa, percent in sorted_aa:
        print(f"  {aa}: {percent:.2f}%")
    
    # Isoelectric point and instability
    isoelectric_point = analyzer.isoelectric_point()
    instability_index = analyzer.instability_index()
    
    print(f"Isoelectric point (pI): {isoelectric_point:.2f}")
    print(f"Instability index: {instability_index:.2f}")
    
    if instability_index < 40:
        print("  Protein is stable")
    else:
        print("  Protein is unstable")
    
    # Hydrophobicity analysis
    hydrophobicity = analyzer.gravy()
    print(f"GRAVY (hydrophobicity): {hydrophobicity:.3f}")
    
    if hydrophobicity > 0:
        print("  Protein is hydrophobic")
    else:
        print("  Protein is hydrophilic")
    
    print("Protein analysis completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"biopython example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: FASTA file handling and sequence I/O

In [ ]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
import shutil
import tempfile
import os

# Example 3: FASTA file handling and sequence I/O
print("Example 3: FASTA file handling and sequence I/O")

try:
    # Create sample sequences
    sequences = [
        SeqRecord(Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"), id="seq1", description="Gene A"),
        SeqRecord(Seq("ATGAAACGCATTAGCACCACCATTACCACCACCATCACCATTACCACAGGTAACGGTGCGGGCTGA"), id="seq2", description="Gene B"),
        SeqRecord(Seq("ATGCGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAATTAGATGGTGATGTT"), id="seq3", description="Gene C")
    ]
    
    print(f"✓ Created {len(sequences)} sample sequences")
    
    # Write sequences to temporary FASTA file
    temp_dir = tempfile.mkdtemp()
    fasta_file = os.path.join(temp_dir, "sample_sequences.fasta")
    
    with open(fasta_file, "w") as output_handle:
        SeqIO.write(sequences, output_handle, "fasta")
    
    print(f"✓ Written sequences to FASTA file")
    
    # Read sequences back from file
    print("Reading sequences from FASTA file:")
    seq_count = 0
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_count += 1
        print(f"  {record.id}: {record.description}")
        print(f"    Length: {len(record.seq)} bp")
        print(f"    Sequence: {str(record.seq)[:30]}...")
        
        # Basic analysis
        gc_content = gc_fraction(record.seq) * 100
        print(f"    GC content: {gc_content:.1f}%")
    
    print(f"✓ Read {seq_count} sequences from file")
    
    # Convert sequences to different formats in memory
    from io import StringIO
    
    # Convert to GenBank format
    genbank_output = StringIO()
    # Add some features for GenBank format
    for i, seq in enumerate(sequences):
        seq.annotations["molecule_type"] = "DNA"
        seq.annotations["organism"] = "Sample organism"
    
    SeqIO.write(sequences, genbank_output, "genbank")
    genbank_content = genbank_output.getvalue()
    
    print(f"✓ Converted to GenBank format ({len(genbank_content)} characters)")
    print(f"GenBank preview: {genbank_content[:100]}...")
    
    # Cleanup
    shutil.rmtree(temp_dir)
    
    print("FASTA file handling completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"biopython example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Sequence alignment and phylogenetic analysis

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import Align
import tempfile

# Example 4: Sequence alignment and phylogenetic analysis
print("Example 4: Sequence alignment and phylogenetic analysis")

try:
    # Create sample sequences for alignment
    sequences = [
        SeqRecord(Seq("ACGTACGTACGTACGT"), id="seq1", description="Sequence 1"),
        SeqRecord(Seq("ACGTACCGTACGTACGT"), id="seq2", description="Sequence 2"),  # One insertion
        SeqRecord(Seq("ACGTACGTACGTCGT"), id="seq3", description="Sequence 3"),   # One deletion
        SeqRecord(Seq("ACGTACGTACGTACGG"), id="seq4", description="Sequence 4")   # One substitution
    ]
    
    print(f"✓ Created {len(sequences)} sequences for alignment")
    
    # Show sequence differences
    print("Sequence comparison:")
    for i, seq in enumerate(sequences, 1):
        print(f"  Seq{i}: {seq.seq} (length: {len(seq.seq)})")
    
    # Pairwise alignment using Biopython's alignment module
    # Bio.pairwise2 is deprecated; PairwiseAligner scored like globalxx (match 1, mismatch 0, gaps free)
    aligner = Align.PairwiseAligner(mode="global", match_score=1, mismatch_score=0, gap_score=0)
    
    # Perform pairwise alignment between first two sequences
    seq1_str = str(sequences[0].seq)
    seq2_str = str(sequences[1].seq)
    
    alignments = aligner.align(seq1_str, seq2_str)
    
    print(f"\nPairwise alignment results:")
    print(f"Found {len(alignments)} alignments")
    # 1 per match, 0 for gaps: seq1 aligns fully inside seq2 (one insertion) -> 16, two optimal alignments
    assert (len(alignments), alignments[0].score) == (2, 16.0), (len(alignments), alignments[0].score)
    
    if alignments:
        best_alignment = alignments[0]
        print(f"Best alignment score: {best_alignment.score}")
        print("Alignment preview:")
        alignment_str = str(best_alignment)
        # Show first 3 lines of alignment
        lines = alignment_str.split('\n')[:3]
        for line in lines:
            if line.strip():
                print(f"  {line}")
    
    # Phylogenetic distance matrix calculation
    print(f"\nPhylogenetic analysis:")
    
    # Calculate pairwise distances (simple Hamming distance)
    distance_matrix = []
    seq_strings = [str(seq.seq) for seq in sequences]
    
    print("Distance matrix:")
    print("     ", end="")
    for i in range(len(sequences)):
        print(f"Seq{i+1:2}", end=" ")
    print()
    
    for i in range(len(sequences)):
        print(f"Seq{i+1:2}:", end=" ")
        row = []
        for j in range(len(sequences)):
            if i == j:
                distance = 0.0
            else:
                # Hamming distance (proportion of differing positions)
                seq_i = seq_strings[i]
                seq_j = seq_strings[j]
                max_len = max(len(seq_i), len(seq_j))
                
                # Pad shorter sequence
                seq_i_padded = seq_i.ljust(max_len, '-')
                seq_j_padded = seq_j.ljust(max_len, '-')
                
                differences = sum(c1 != c2 for c1, c2 in zip(seq_i_padded, seq_j_padded))
                distance = differences / max_len
            
            row.append(distance)
            print(f"{distance:.2f}", end="  ")
        print()
        distance_matrix.append(row)
    
    # Find most similar pair
    min_distance = float('inf')
    most_similar_pair = None
    
    for i in range(len(sequences)):
        for j in range(i + 1, len(sequences)):
            if distance_matrix[i][j] < min_distance:
                min_distance = distance_matrix[i][j]
                most_similar_pair = (i + 1, j + 1)
    
    if most_similar_pair:
        print(f"\nMost similar sequences: Seq{most_similar_pair[0]} and Seq{most_similar_pair[1]}")
        print(f"Distance: {min_distance:.2f}")
    
    print("Sequence alignment analysis completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"biopython example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Package capabilities and bioinformatics applications

In [ ]:
import Bio
from Bio.Seq import Seq
from Bio.SeqUtils import molecular_weight, gc_fraction

# Example 5: Biopython package capabilities and bioinformatics applications
print("Example 5: Biopython package capabilities and bioinformatics applications")

try:
    print("Biopython - Comprehensive bioinformatics toolkit for Python")
    
    # Show package version
    print(f"Biopython version: {Bio.__version__}")
    
    # Key Biopython capabilities
    print("\nKey Biopython capabilities:")
    capabilities = [
        "🧬 Sequence manipulation and analysis (DNA, RNA, protein)",
        "🔍 Sequence alignment algorithms and tools", 
        "📊 Phylogenetic analysis and tree construction",
        "💾 File format support (FASTA, GenBank, PDB, etc.)",
        "🌐 Database access (NCBI, UniProt, PDB)",
        "🧪 Restriction enzyme analysis",
        "📈 Population genetics and evolution analysis",
        "🔬 Structure analysis and visualization support"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Major Biopython modules
    print("\nMajor Biopython modules:")
    modules = [
        ("Bio.Seq", "Core sequence objects and operations"),
        ("Bio.SeqIO", "Sequence input/output in multiple formats"),
        ("Bio.SeqRecord", "Sequence annotation and metadata"),
        ("Bio.SeqUtils", "Sequence utility functions and analysis"),
        ("Bio.Align", "Multiple sequence alignment tools"),
        ("Bio.Phylo", "Phylogenetic analysis and tree handling"),
        ("Bio.PDB", "Protein structure analysis"),
        ("Bio.Blast", "BLAST sequence similarity search"),
        ("Bio.Entrez", "NCBI database access utilities"),
        ("Bio.SeqFeature", "Sequence feature annotation")
    ]
    
    for module, description in modules:
        print(f"  • {module}: {description}")
    
    # File format support
    print("\nSupported file formats:")
    formats = [
        "FASTA - sequence data", "GenBank - annotated sequences", 
        "EMBL - European sequence format", "SwissProt - protein database format",
        "PDB - protein structure data", "Stockholm - multiple alignment format",
        "Phylip - phylogenetic data", "Nexus - evolutionary analysis format",
        "BLAST - similarity search results", "Clustal - alignment program output"
    ]
    
    for fmt in formats:
        print(f"  • {fmt}")
    
    # Common use cases and applications
    print("\nCommon bioinformatics applications:")
    applications = [
        "🔬 Genomic sequence analysis and annotation",
        "🧪 Protein structure and function prediction", 
        "🌿 Phylogenetic reconstruction and evolution",
        "💊 Drug discovery and molecular design",
        "🦠 Pathogen identification and epidemiology",
        "🌾 Agricultural genomics and breeding",
        "🏥 Clinical diagnostics and personalized medicine",
        "🧬 Comparative genomics and metagenomics"
    ]
    
    for app in applications:
        print(f"  {app}")
    
    # Integration with scientific ecosystem
    print("\nIntegration with scientific Python ecosystem:")
    integrations = [
        "• NumPy: Numerical operations on biological data",
        "• Pandas: Biological data analysis and manipulation", 
        "• Matplotlib: Visualization of biological data",
        "• SciPy: Statistical analysis of biological systems",
        "• Jupyter: Interactive bioinformatics notebooks",
        "• scikit-learn: Machine learning on biological data"
    ]
    
    for integration in integrations:
        print(f"  {integration}")
    
    # Simple demonstration of key features
    print(f"\nKey features demonstration:")
    
    # 1. Sequence manipulation
    dna = Seq("ATGGCCATTGTAATG")
    protein = dna.translate()
    assert str(protein) == "MAIVM", protein
    print(f"  ✓ Translation: {dna} -> {protein}")
    
    # 2. Sequence analysis
    gc_content = gc_fraction(dna) * 100  # Convert to percentage
    mol_weight = molecular_weight(protein, seq_type="protein")
    assert (round(gc_content, 1), round(mol_weight, 1)) == (40.0, 563.8), (gc_content, mol_weight)
    print(f"  ✓ Analysis: GC content {gc_content:.1f}%, protein MW {mol_weight:.1f} Da")
    
    # 3. Format flexibility
    from Bio.SeqRecord import SeqRecord
    record = SeqRecord(dna, id="sample", description="Demo sequence")
    print(f"  ✓ SeqRecord: {record.id} - {record.description}")
    
    # 4. Multiple sequence types
    rna = dna.transcribe()
    rev_comp = dna.reverse_complement()
    print(f"  ✓ RNA: {rna}")
    assert (str(rna), str(rev_comp)) == ("AUGGCCAUUGUAAUG", "CATTACAATGGCCAT"), (rna, rev_comp)
    print(f"  ✓ Reverse complement: {rev_comp}")
    
    print(f"\nBiopython enables professional bioinformatics research")
    print("Biopython example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"biopython example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")